In [30]:
import pandas as pd
import numpy as np
from win_loss_utils import c1, c2, c3, c1_argmax_n2, c2_argmax_n2, c3_argmax_n2_fast
import ast

In [21]:
## Load and prepare the three datasets

# Load raw data
soccer = pd.read_csv("../../data/processed/soccer.csv")
soccer_binary = pd.read_csv("../../data/processed/soccer_binary.csv")
us_leagues = pd.read_csv("../../data/processed/us_leagues.csv")

# Add type column and select relevant columns
soccer['type'] = 'soccer_ternary'
soccer_binary['type'] = 'soccer_binary'
us_leagues['type'] = 'us'

# Standardize columns
soccer = soccer[['type', 'League', 'Season', 'Team', 'Sequence']]
soccer_binary = soccer_binary[['type', 'League', 'Season', 'Team', 'Sequence']]
us_leagues = us_leagues[['type', 'League', 'Season', 'Team', 'Sequence']]

# Rename columns to match schema
soccer.columns = ['type', 'league', 'season', 'team', 'sequence']
soccer_binary.columns = ['type', 'league', 'season', 'team', 'sequence']
us_leagues.columns = ['type', 'league', 'season', 'team', 'sequence']

# Convert sequence strings to lists
soccer['sequence'] = soccer['sequence'].apply(ast.literal_eval)
soccer_binary['sequence'] = soccer_binary['sequence'].apply(ast.literal_eval)
us_leagues['sequence'] = us_leagues['sequence'].apply(ast.literal_eval)

# Combine all datasets
master = pd.concat([soccer, soccer_binary, us_leagues], ignore_index=True)

print(f"Master dataset shape: {master.shape}")
print(f"Types: {master['type'].unique()}")
print(f"\nFirst few rows:")
print(master.head())

Master dataset shape: (6771, 5)
Types: ['soccer_ternary' 'soccer_binary' 'us']

First few rows:
             type      league  season           team  \
0  soccer_ternary  Bundesliga    2000  Bayern Munich   
1  soccer_ternary  Bundesliga    2000         Bochum   
2  soccer_ternary  Bundesliga    2000        Cottbus   
3  soccer_ternary  Bundesliga    2000       Dortmund   
4  soccer_ternary  Bundesliga    2000  Ein Frankfurt   

                                            sequence  
0  [3, 3, 3, 0, 3, 3, 0, 0, 3, 1, 3, 0, 0, 1, 3, ...  
1  [3, 0, 0, 0, 3, 1, 3, 0, 0, 0, 1, 0, 1, 0, 3, ...  
2  [0, 0, 0, 3, 0, 0, 1, 3, 0, 3, 1, 3, 0, 0, 3, ...  
3  [3, 3, 0, 3, 3, 0, 1, 3, 0, 0, 0, 3, 1, 3, 3, ...  
4  [3, 0, 3, 0, 3, 1, 1, 0, 0, 3, 0, 3, 3, 0, 0, ...  


In [22]:
# Add sequence length column
master['n'] = master['sequence'].apply(len)
print("Added sequence length 'n'")


Added sequence length 'n'


In [23]:
## Compute C1 metrics (binary sequences only)

# Add c1 score with n2 = L // 2
def compute_c1_score(row):
    if row['type'] == 'soccer_ternary':
        return np.nan
    seq = row['sequence']
    n2_default = len(seq) // 2
    return c1(seq, n2=n2_default)

master['c1'] = master.apply(compute_c1_score, axis=1)
print("Added c1 score (n2 = L // 2)") 

# Add n1* and c1*
def compute_c1_argmax_n2(row):
    """Compute c1* and n1* only for non-ternary sequences"""
    if row['type'] == 'soccer_ternary':
        return np.nan, np.nan
    seq = row['sequence']
    n2_default = len(seq) // 2
    n1_star, c1_star = c1_argmax_n2(seq, n2_default)
    return n1_star, c1_star

results = [compute_c1_argmax_n2(row) for _, row in master.iterrows()]
master['n1*'], master['c1*'] = zip(*results)
print("Added n1*, c1*")


Added c1 score (n2 = L // 2)
Added n1*, c1*


In [45]:
## Compute C2 metrics

# Add c2 score with n2 = L // 2

def compute_c2_score(row):
    seq = row['sequence']
    n2_default = len(seq) // 2
    return c2(seq, n2=n2_default)

master['c2'] = master.apply(compute_c2_score, axis=1)
print("Added c2 score (n2 = L // 2)")

# Add n2* and c2*

def compute_c2_argmax_n2(row):
    seq = row['sequence']
    n2_default = len(seq) // 2
    n2_star, c2_star = c2_argmax_n2(seq, n2_default)
    return n2_star, c2_star

results = [compute_c2_argmax_n2(row) for _, row in master.iterrows()]
master['n2*'], master['c2*'] = zip(*results)
print("Added n2*, c2*")

C:\Users\ryanj\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\scipy\stats\_axis_nan_policy.py:573: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Added c2 score (n2 = L // 2)
Added n2*, c2*


In [25]:
## Compute C3 metrics (initial pass with N=1000)

# Add c3 score with n2 = L // 2
master['c3'] = [c3(np.array(seq), N=1000, n2=len(seq)//2) for seq in master['sequence']]
print("Added c3 score with N=1000")


## Add n3* and c3*
results = []
for _, row in master.iterrows():
    seq = np.array(row['sequence'])
    n3_star, c3_star = c3_argmax_n2_fast(seq, N=1000)
    results.append((n3_star, c3_star))

master['n3*'], master['c3*'] = zip(*results)
print("Added n3*, c3* with N=1000")


Added c3 score with N=1000
Added n3*, c3* with N=1000


In [26]:
## Recalculate top 100 C3 scores with higher accuracy (N=100000)

N_high = 100000

# Recalculate for each type separately
for type_name in master['type'].unique():
    print(f"\nRecalculating top 100 C3 for {type_name}...")
    
    # Get indices of top 100 for this type
    mask = master['type'] == type_name
    type_data = master[mask]
    top_100_indices = type_data.nlargest(100, 'c3*').index
    
    # Recalculate with higher accuracy
    for idx in top_100_indices:
        seq = np.array(master.loc[idx, 'sequence'])
        n3_star, c3_star = c3_argmax_n2_fast(seq, N=N_high)
        master.loc[idx, 'n3*'] = n3_star
        master.loc[idx, 'c3*'] = c3_star
        master.loc[idx, 'c3'] = c3(seq, N=N_high, n2=len(seq)//2)



Recalculating top 100 C3 for soccer_ternary...

Recalculating top 100 C3 for soccer_binary...

Recalculating top 100 C3 for us...


In [46]:
## Finalize the master database with correct schema

# Reorder and select only the required columns
final_columns = ['type', 'league', 'season', 'team', 'sequence', 
                 'n', 'c1', 'c2', 'c3', 
                 'n1*', 'c1*', 'n2*', 'c2*', 'n3*', 'c3*']

master_db = master[final_columns].copy()

print("Final Master Database Schema:")
print(f"Columns: {master_db.columns.tolist()}")
print(f"Shape: {master_db.shape}")
print(f"\nData types:")
print(master_db.dtypes)
print(f"\nSummary by type:")
print(master_db.groupby('type').size())

Final Master Database Schema:
Columns: ['type', 'league', 'season', 'team', 'sequence', 'n', 'c1', 'c2', 'c3', 'n1*', 'c1*', 'n2*', 'c2*', 'n3*', 'c3*']
Shape: (6771, 15)

Data types:
type         object
league       object
season        int64
team         object
sequence     object
n             int64
c1          float64
c2          float64
c3          float64
n1*         float64
c1*         float64
n2*         float64
c2*         float64
n3*           int64
c3*         float64
dtype: object

Summary by type:
type
soccer_binary     1430
soccer_ternary    1430
us                3911
dtype: int64


In [47]:
## Save the master database

# Convert sequence lists back to strings for CSV storage
master_db_for_csv = master_db.copy()
master_db_for_csv['sequence'] = master_db_for_csv['sequence'].apply(str)

# Save to CSV
output_path = "../../output/csvs/master_database.csv"
master_db_for_csv.to_csv(output_path, index=False)
print(f"Master database saved to: {output_path}")
print(f"Total records: {len(master_db_for_csv)}")

Master database saved to: ../../output/csvs/master_database.csv
Total records: 6771


In [48]:
master_db

,type,league,season,team,sequence,n,c1,c2,c3,n1*,c1*,n2*,c2*,n3*,c3*
0,soccer_ternary,Bundesliga,2000,Bayern Munich,"[3, 3, 3, 0, 3, 3, 0, 0, 3, 1, 3, 0, 0, 1, 3, ...",35,NaN,2.260805,2.087683,NaN,NaN,2.0,11.787903,2,6.289308
1,soccer_ternary,Bundesliga,2000,Bochum,"[3, 0, 0, 0, 3, 1, 3, 0, 0, 0, 1, 0, 1, 0, 3, ...",34,NaN,9.841874,6.535948,NaN,NaN,6.0,12.382271,17,8.333333
2,soccer_ternary,Bundesliga,2000,Cottbus,"[0, 0, 0, 3, 0, 0, 1, 3, 0, 3, 1, 3, 0, 0, 3, ...",35,NaN,1.551888,1.455604,NaN,NaN,15.0,1.688276,15,1.477105
3,soccer_ternary,Bundesliga,2000,Dortmund,"[3, 3, 0, 3, 3, 0, 1, 3, 0, 0, 0, 3, 1, 3, 3, ...",35,NaN,2.957636,2.444988,NaN,NaN,16.0,4.381153,16,3.937008
4,soccer_ternary,Bundesliga,2000,Ein Frankfurt,"[3, 0, 3, 0, 3, 1, 1, 0, 0, 3, 0, 3, 3, 0, 0, ...",34,NaN,3.776935,2.808989,NaN,NaN,12.0,8.047010,13,6.849315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6766,us,NBA,1982,Indiana Pacers,"[1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, ...",82,5.464184,15.885321,11.494253,37.0,12.046809,37.0,52.729340,37,29.411765
6767,us,NBA,1982,Washington Bullets,"[0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, ...",82,1.158671,1.104617,1.071811,3.0,2.075934,1.0,6.484131,3,2.074689
6768,us,NBA,1982,Dallas Mavericks,"[1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, ...",82,1.774389,2.000000,1.718213,11.0,18.279912,11.0,44.700115,11,23.809524
6769,us,NBA,1982,Denver Nuggets,"[0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, ...",82,1.115894,1.065703,1.037344,15.0,1.812723,15.0,2.232645,15,1.818182
